# 01 - Data Quality

**Objective:** Profile the raw Sample Superstore dataset and document its actual
structure, before any cleaning or analysis decisions are made.

**Business context:** This project builds a Sales Performance & Profitability
Analytics solution for a fictional retail company using the public **Sample
Superstore** dataset (a well-known dataset used for BI/analytics learning,
not real transactional data from any real company).

This notebook answers: *is the data trustworthy enough to build KPIs and a
dashboard on top of it?*


In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
from data_loading import load_orders, load_people, load_returns
from data_cleaning import clean_orders, standardize_column_names
from validation import run_all_checks
import os
from pathlib import Path

print("Current working directory:")
print(os.getcwd())

print("\nExpected project root:")
print(Path.cwd().parent)

print("\nRaw data from current directory:")
print(Path("data/raw").resolve())
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

raw = load_orders()
people = load_people()
returns = load_returns()
print(f"Orders: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"People: {people.shape if people is not None else 'not found'}")
print(f"Returns: {returns.shape if returns is not None else 'not found'}")


Current working directory:
c:\Users\syedm\Downloads\sales-performance-profitability-analytics\sales-performance-profitability-analytics\notebooks

Expected project root:
c:\Users\syedm\Downloads\sales-performance-profitability-analytics\sales-performance-profitability-analytics

Raw data from current directory:
C:\Users\syedm\Downloads\sales-performance-profitability-analytics\sales-performance-profitability-analytics\notebooks\data\raw
Orders: 10,194 rows x 21 columns
People: (4, 2)
Returns: (296, 2)


## Column overview and data types

In [2]:
raw.dtypes.to_frame('dtype')

,dtype
Row ID,int64
Order ID,object
Order Date,datetime64[ns]
Ship Date,datetime64[ns]
Ship Mode,object
Customer ID,object
Customer Name,object
Segment,object
Country/Region,object
City,object


## Missing values, duplicates, and date range

None of these are assumptions -- every number below is calculated directly
from the file.

In [3]:
df = standardize_column_names(raw)

print("Missing values per column:")
missing = df.isna().sum()
print(missing[missing > 0] if (missing > 0).any() else "None found.")

print(f"\nFully duplicated rows: {df.duplicated().sum()}")
print(f"Duplicate row_id values: {df['row_id'].duplicated().sum()}")
print(f"\nOrder Date range: {df['order_date'].min()} to {df['order_date'].max()}")
print(f"Ship Date range:  {df['ship_date'].min()} to {df['ship_date'].max()}")


Missing values per column:
None found.

Fully duplicated rows: 0
Duplicate row_id values: 0

Order Date range: 2023-01-03 00:00:00 to 2026-12-30 00:00:00
Ship Date range:  2023-01-07 00:00:00 to 2027-01-05 00:00:00


## Cardinality: customers, products, categories, regions

In [4]:
for col in ['order_id','customer_id','product_id','category','sub_category','region','state_province','segment','country_region']:
    print(f"{col:>18}: {df[col].nunique():,} distinct values")


          order_id: 5,111 distinct values
       customer_id: 804 distinct values
        product_id: 1,862 distinct values
          category: 3 distinct values
      sub_category: 17 distinct values
            region: 4 distinct values
    state_province: 59 distinct values
           segment: 3 distinct values
    country_region: 2 distinct values


## Numeric ranges (Sales, Quantity, Discount, Profit)

Checking for the classic data-quality red flags: negative/zero sales,
negative quantities, and out-of-range discounts.

In [5]:
display_df = df[['sales','quantity','discount','profit']].describe().T
display_df

,count,mean,std,min,25%,50%,75%,max
sales,10194.0,228.225854,619.906839,0.444,17.2200,53.91,209.500000,22638.480
quantity,10194.0,3.791838,2.228317,1.000,2.0000,3.00,5.000000,14.000
discount,10194.0,0.155385,0.206249,0.000,0.0000,0.20,0.200000,0.800
profit,10194.0,28.673417,232.465115,-6599.978,1.7608,8.69,29.297925,8399.976


In [6]:
print("Rows with sales <= 0:   ", (df['sales'] <= 0).sum())
print("Rows with quantity <= 0:", (df['quantity'] <= 0).sum())
print("Rows with discount outside [0,1]:", ((df['discount']<0)|(df['discount']>1)).sum())


Rows with sales <= 0:    0
Rows with quantity <= 0: 0
Rows with discount outside [0,1]: 0


## Known data quirks (documented, not silently fixed)

Two things stood out during profiling that are worth flagging explicitly:
1. A handful of `order_id`s map to more than one `customer_id`.
2. One customer ("Harry Olson", not shown here -- see docs/data_quality.md)
   appears under 4 different Customer IDs.

Both are treated as **known source-system quirks** and documented rather
than silently corrected, since there's no reliable way to know which ID is
"right".

In [7]:
multi_customer_orders = df.groupby('order_id')['customer_id'].nunique()
print(f"order_ids spanning multiple customer_ids: {(multi_customer_orders > 1).sum()}")

multi_name_products = df.groupby('product_id')['product_name'].nunique()
print(f"product_ids with more than one product_name: {(multi_name_products > 1).sum()}")


order_ids spanning multiple customer_ids: 2
product_ids with more than one product_name: 32


## Referential integrity: Returns -> Orders

In [8]:
orphan_returns = (~returns['Order ID'].isin(df['order_id'])).sum()
print(f"Returns rows referencing an unknown Order ID: {orphan_returns} / {len(returns)}")
print(f"Return rate (orders with at least one returned line): {returns['Order ID'].nunique() / df['order_id'].nunique():.2%}")


Returns rows referencing an unknown Order ID: 0 / 296
Return rate (orders with at least one returned line): 5.79%


## Running the full validation suite (src/validation.py)

In [9]:
results = run_all_checks(df)
for r in results:
    print(r)
passed = sum(r.passed for r in results)
print(f"\n{passed}/{len(results)} checks passed.")


[PASS] required_columns: all required columns present
[PASS] no_missing_values: no missing values
[PASS] no_full_duplicate_rows: no duplicate rows
[PASS] row_id_is_unique: row_id is unique
[PASS] dates_valid: all dates parse and ship_date >= order_date for every row
[PASS] positive_quantity: all quantities > 0
[PASS] positive_sales: all sales values > 0
[PASS] discount_in_valid_range: all discounts in [0, 1]
[FAIL] order_maps_to_single_customer: 2 order_id(s) span multiple customer_id values (documented known quirk, see docs/data_quality.md)

8/9 checks passed.


## Findings

- The dataset has **zero missing values, zero full-row duplicates, and zero
  invalid numeric values** (no non-positive sales/quantity, no out-of-range
  discounts). This is a clean dataset at the row level.
- It spans **2023-01-03 to 2026-12-30**, 2 countries (United States, Canada),
  4 regions, 3 categories, 17 sub-categories, 804 customers, and 1,862
  distinct product IDs.
- The only validation failure is a **known, narrow quirk**: 2 order_ids
  span multiple customer_ids, traced to one customer ("Harry Olson")
  appearing under 4 different Customer IDs. This does not block analysis --
  it's documented in `docs/data_quality.md` and pinned by a regression test
  in `tests/test_data_quality.py`.
- **Verdict: the dataset is suitable for the full analysis as scoped**, with
  one real constraint: there is no unit-cost column, so only a
  revenue-based profit margin (Profit / Sales) can be calculated -- see
  `docs/assumptions_and_constraints.md`.
